# 05 Models — XGBoost — Men's

XGBoost is the consensus top performer for this competition (research). We use shallow trees with regularization and LOGO-CV for honest evaluation.

**Approach:**
- LOGO cross-validation (each season = one fold)
- Train on all other seasons, predict held-out season's tournament games
- Evaluate with Brier score (MSE of predicted probabilities vs actual outcomes)
- Train final model on all data for Stage 1/2 predictions
- Apply Platt scaling (isotonic regression) for calibration

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/xgboost/mens/`):
- `oof_predictions.parquet` — out-of-fold predictions for all training matchups
- `stage1_predictions.parquet` — predictions for Stage 1 grid
- `stage2_predictions.parquet` — predictions for Stage 2 grid
- `cv_results.parquet` — per-fold Brier scores

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except:
    !pip install 'xgboost>=2.0,<3.0'
    import xgboost as xgb

try:
    import optuna
except:
    !pip install optuna
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"XGBoost version: {xgb.__version__}")
print(f"Optuna version: {optuna.__version__}")

XGBoost version: 2.1.4
Optuna version: 4.7.0


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

def brier_objective(predt, dtrain):
    """Custom Brier score objective for XGBoost.
    Gradient and hessian of Brier loss w.r.t. raw predictions (before sigmoid).
    """
    labels = dtrain.get_label()
    preds = 1.0 / (1.0 + np.exp(-predt))  # sigmoid
    
    # Gradient: d(Brier)/d(raw) = 2(p - y) * p(1-p)
    grad = 2.0 * (preds - labels) * preds * (1.0 - preds)
    
    # Hessian: d²(Brier)/d(raw)²
    hess = 2.0 * preds * (1.0 - preds) * (preds * (1.0 - preds) + (preds - labels) * (1.0 - 2.0 * preds))
    # Ensure hessian is positive for stability
    hess = np.maximum(hess, 1e-6)
    
    return grad, hess

def brier_eval(predt, dtrain):
    """Custom Brier score evaluation metric for XGBoost."""
    labels = dtrain.get_label()
    preds = 1.0 / (1.0 + np.exp(-predt))  # sigmoid
    brier = np.mean((preds - labels) ** 2)
    return 'brier', brier

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/xgboost"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (5170, 209)
Stage 1 grid: (261013, 207)
Stage 2 grid: (66430, 206)
Features: 70


## 2. Hyperparameter Tuning with Optuna

Bayesian optimization over XGBoost hyperparameters using LOGO-CV Brier score as the objective. Tunes on Stage 1 validation years (2022–2025, 4 folds) for speed — these are the most relevant years and match the competition's scoring window.

In [6]:
def optuna_objective(trial):
    """Optuna objective: returns mean LOGO-CV Brier score on Stage 1 folds."""
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'seed': 42,
        'verbosity': 0,
        'disable_default_eval_metric': True,
    }
    
    n_rounds = 1000
    early_stopping = 50
    
    # Use Stage 1 validation folds (2022-2025) for fast, relevant tuning
    tune_folds = [2022, 2023, 2024, 2025]
    train_original_tmp = train[train['TeamA'] < train['TeamB']].copy()
    
    fold_briers = []
    for fold in tune_folds:
        train_mask = (train['Fold'] != fold)
        val_mask = (train_original_tmp['Fold'] == fold)
        
        X_tr = train.loc[train_mask, feature_cols]
        y_tr = train.loc[train_mask, 'Label']
        X_va = train_original_tmp.loc[val_mask, feature_cols]
        y_va = train_original_tmp.loc[val_mask, 'Label']
        
        if len(X_va) == 0:
            continue
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_va, label=y_va)
        
        model = xgb.train(
            params, dtrain, num_boost_round=n_rounds,
            obj=brier_objective,
            evals=[(dval, 'val')],
            custom_metric=brier_eval,
            early_stopping_rounds=early_stopping,
            verbose_eval=False
        )
        
        raw_preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
        preds = 1.0 / (1.0 + np.exp(-raw_preds))
        fold_briers.append(brier_score_loss(y_va, preds))
    
    return np.mean(fold_briers)

# Run optimization — 50 trials with 4 folds each
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest trial Brier: {study.best_value:.4f}")
print(f"Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]


Best trial Brier: 0.1816
Best hyperparameters:
  max_depth: 2
  learning_rate: 0.27051668818999286
  subsample: 0.8875664116805573
  colsample_bytree: 0.9697494707820946
  min_child_weight: 18
  reg_alpha: 0.6218704727769077
  reg_lambda: 5.829384542994738


In [7]:
best = study.best_params

# best = {
#     'max_depth': 2,
#     'learning_rate': 0.0.27051668818999286,
#     'subsample': 0.0.8875664116805573,
#     'colsample_bytree': 0.0.9697494707820946,
#     'min_child_weight': 18,
#     'reg_alpha': 0.0.6218704727769077,
#     'reg_lambda': 0.5.829384542994738,
# }

In [8]:
# Build final parameter dict from Optuna results
xgb_params = {
    **best,
    # **study.best_params,
    'seed': 42,
    'verbosity': 0,
    'disable_default_eval_metric': True,
}

N_ROUNDS = 1000
EARLY_STOPPING = 50

print("Tuned XGBoost parameters:")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")
print(f"  n_rounds: {N_ROUNDS}")
print(f"  early_stopping: {EARLY_STOPPING}")

Tuned XGBoost parameters:
  max_depth: 2
  learning_rate: 0.27051668818999286
  subsample: 0.8875664116805573
  colsample_bytree: 0.9697494707820946
  min_child_weight: 18
  reg_alpha: 0.6218704727769077
  reg_lambda: 5.829384542994738
  seed: 42
  verbosity: 0
  disable_default_eval_metric: True
  n_rounds: 1000
  early_stopping: 50


## 3. LOGO Cross-Validation

Train on all folds except one, predict the held-out fold. Collect out-of-fold predictions for all training matchups.

In [9]:
# Only use the original (non-flipped) rows for OOF evaluation
# The flipped rows are for training only
# Original rows have TeamA < TeamB (submission format)
train_original = train[train['TeamA'] < train['TeamB']].copy()

folds = sorted(train['Fold'].unique())
print(f"Total folds: {len(folds)}")
print(f"Original (non-flipped) matchups: {len(train_original)}")

Total folds: 40
Original (non-flipped) matchups: 2585


In [10]:
oof_preds = []
cv_results = []

for fold in folds:
    # Train on all folds except current (use flip-doubled data for training)
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        obj=brier_objective,
        evals=[(dval, 'val')],
        custom_metric=brier_eval,
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False
    )
    
    # With custom objective, predict returns raw scores — apply sigmoid
    raw_preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    preds = 1.0 / (1.0 + np.exp(-raw_preds))
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1985: Brier=0.1836, Games=63, BestRound=38
  Fold 1986: Brier=0.2083, Games=63, BestRound=35
  Fold 1987: Brier=0.1821, Games=63, BestRound=23
  Fold 1988: Brier=0.1912, Games=63, BestRound=118
  Fold 1989: Brier=0.1729, Games=63, BestRound=27
  Fold 1990: Brier=0.1933, Games=63, BestRound=70
  Fold 1991: Brier=0.1829, Games=63, BestRound=97
  Fold 1992: Brier=0.1730, Games=63, BestRound=58
  Fold 1993: Brier=0.1630, Games=63, BestRound=53
  Fold 1994: Brier=0.1674, Games=63, BestRound=38
  Fold 1995: Brier=0.1674, Games=63, BestRound=21
  Fold 1996: Brier=0.1729, Games=63, BestRound=22
  Fold 1997: Brier=0.2009, Games=63, BestRound=57
  Fold 1998: Brier=0.1898, Games=63, BestRound=8
  Fold 1999: Brier=0.1906, Games=63, BestRound=94
  Fold 2000: Brier=0.1934, Games=63, BestRound=18
  Fold 2001: Brier=0.1943, Games=64, BestRound=9
  Fold 2002: Brier=0.1864, Games=64, BestRound=42
  Fold 2003: Brier=0.1793, Games=64, BestRound=55
  Fold 2004: Brier=0.1778, Games=64, BestRound=25
 

In [11]:
# Overall CV Brier score
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"\nCV Results Summary:")
print(f"  Mean Brier: {cv_df['BrierScore'].mean():.4f}")
print(f"  Std Brier: {cv_df['BrierScore'].std():.4f}")
print(f"  Min Brier: {cv_df['BrierScore'].min():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmin(), 'Fold']})")
print(f"  Max Brier: {cv_df['BrierScore'].max():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmax(), 'Fold']})")


Overall OOF Brier Score: 0.1816
Stage 1 (2022-2025) Brier Score: 0.1816

CV Results Summary:
  Mean Brier: 0.1815
  Std Brier: 0.0179
  Min Brier: 0.1404 (Fold 2025)
  Max Brier: 0.2096 (Fold 2022)


## 4. Train Final Model

Train on all data for generating Stage 1 and Stage 2 predictions. Use the median best round from CV as the number of boosting rounds.

In [12]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds} (median of CV best rounds)")

X_all = train[feature_cols]
y_all = train['Label']

dtrain_all = xgb.DMatrix(X_all, label=y_all)
final_model = xgb.train(xgb_params, dtrain_all, num_boost_round=final_rounds, obj=brier_objective)

Final model rounds: 38 (median of CV best rounds)


## 5. Calibration with Isotonic Regression

XGBoost probabilities often cluster around 0.5. We fit isotonic regression on OOF predictions to improve calibration, then apply it to final predictions.

In [13]:
# Fit calibrator on OOF predictions
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

# Check calibration improvement on OOF
oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])

print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")
print(f"Improvement: {overall_brier - calibrated_brier:+.4f}")

OOF Brier (raw): 0.1816
OOF Brier (calibrated): 0.1783
Improvement: +0.0033


## 6. Generate Predictions

In [14]:
# Stage 1 predictions — apply sigmoid to raw scores
dstage1 = xgb.DMatrix(stage1[feature_cols])
raw_preds_s1 = final_model.predict(dstage1)
stage1['Pred_xgboost'] = 1.0 / (1.0 + np.exp(-raw_preds_s1))
stage1['Pred_xgboost_calibrated'] = calibrator.predict(stage1['Pred_xgboost'].values)

# Clip to avoid extreme probabilities
stage1['Pred_xgboost_calibrated'] = stage1['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 1 predictions: {stage1.shape}")
print(f"  Pred range: [{stage1['Pred_xgboost_calibrated'].min():.3f}, {stage1['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage1['Pred_xgboost_calibrated'].mean():.3f}")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier_raw = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost'])
    s1_brier_cal = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost_calibrated'])
    print(f"\n  Stage 1 Brier (raw): {s1_brier_raw:.4f}")
    print(f"  Stage 1 Brier (calibrated): {s1_brier_cal:.4f}")

Stage 1 predictions: (261013, 209)
  Pred range: [0.020, 0.980]
  Pred mean: 0.300

  Stage 1 Brier (raw): 0.1761
  Stage 1 Brier (calibrated): 0.1757


In [15]:
# Stage 2 predictions — apply sigmoid to raw scores
dstage2 = xgb.DMatrix(stage2[feature_cols])
raw_preds_s2 = final_model.predict(dstage2)
stage2['Pred_xgboost'] = 1.0 / (1.0 + np.exp(-raw_preds_s2))
stage2['Pred_xgboost_calibrated'] = calibrator.predict(stage2['Pred_xgboost'].values)
stage2['Pred_xgboost_calibrated'] = stage2['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 2 predictions: {stage2.shape}")
print(f"  Pred range: [{stage2['Pred_xgboost_calibrated'].min():.3f}, {stage2['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage2['Pred_xgboost_calibrated'].mean():.3f}")

Stage 2 predictions: (66430, 208)
  Pred range: [0.020, 0.795]
  Pred mean: 0.292


## 7. Feature Importance

In [16]:
importance = final_model.get_score(importance_type='gain')
imp_df = pd.DataFrame({'Feature': importance.keys(), 'Gain': importance.values()})
imp_df = imp_df.sort_values('Gain', ascending=False)

print("Feature Importance (gain):")
for _, row in imp_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Gain']:.4f}")

Feature Importance (gain):
  SeedDiff                      : 254.4570
  TopSystemsAvgRankDiff         : 20.0068
  EloDiff                       : 14.2358
  Seed_x_Rank                   : 13.5741
  IsPowerConfDiff               : 10.2286
  SeedB                         : 5.5569
  SOSDiff                       : 5.3041
  FGM_MarginDiff                : 4.6731
  Stl_MarginDiff                : 3.7844
  AvgPointDiffDiff              : 3.6460
  FTPctDiff                     : 3.3071
  HomeWinPctDiff                : 3.2423
  FTA_MarginDiff                : 3.1643
  PowerConfWinPctDiff           : 2.7454
  WinPctDiff                    : 2.6676
  TotalReb_MarginDiff           : 2.6409
  AvgTODiff                     : 2.5508
  Top5WinPctDiff                : 2.5209
  AvgStlDiff                    : 2.4525
  FTrDiff                       : 2.4409
  AvgORDiff                     : 2.3959
  FGM3_MarginDiff               : 2.3236
  FGPctDiff                     : 2.3038
  TO_MarginDiff         

## 8. Save Outputs

In [17]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/cv_results.parquet ((40, 4))


## 9. Summary

In [18]:
print("=" * 60)
print("XGBOOST MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
if stage1_brier:
    print(f"Stage 1 Brier Score: {stage1_brier:.4f}")
print(f"\nCV Folds: {len(cv_df)}")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")
print(f"\nTop 5 features (gain):")
for _, row in imp_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Gain']:.4f}")

XGBOOST MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.1816
OOF Brier Score (calibrated): 0.1783
Stage 1 Brier Score: 0.1816

CV Folds: 40
Mean Brier: 0.1815 +/- 0.0179
Final model rounds: 38

Top 5 features (gain):
  SeedDiff: 254.4570
  TopSystemsAvgRankDiff: 20.0068
  EloDiff: 14.2358
  Seed_x_Rank: 13.5741
  IsPowerConfDiff: 10.2286
